In [152]:
%pip install -q nfl-data-py

import pandas as pd
import numpy as np
import nfl_data_py as nfl


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [153]:
from datetime import datetime

# Set the season to analyze (e.g., 2024). Change if needed.
season = 2025

# Load play-by-play data for the chosen season
pbp = nfl.import_pbp_data(years=[2025])

# Filter for regular season, Week 1
week1 = pbp[(pbp['week'] == 1)]

# Keep only touchdown plays
week1_tds = week1[week1['touchdown'] == 1]

# Count TDs per scorer. Prefer id+name if both available, else fall back to name only
use_cols = [c for c in ['td_player_id', 'td_player_name'] if c in week1_tds.columns]
if use_cols:
    scorers = (
        week1_tds.dropna(subset=use_cols)
        .groupby(use_cols)
        .size()
        .reset_index(name='tds')
    )
    if 'td_player_id' in use_cols:
        scorers = scorers.rename(columns={'td_player_name': 'player', 'td_player_id': 'player_id'})
    else:
        scorers = scorers.rename(columns={'td_player_name': 'player'})
else:
    # Fallback if td_* columns not present; derive from rusher/receiver
    rush = week1_tds.dropna(subset=['rusher_player_id'])[['rusher_player_id', 'rusher_player_name']]
    rec = week1_tds.dropna(subset=['receiver_player_id'])[['receiver_player_id', 'receiver_player_name']]
    rush.columns = ['player_id', 'player']
    rec.columns = ['player_id', 'player']
    both = pd.concat([rush, rec], ignore_index=True)
    scorers = both.groupby(['player_id', 'player']).size().reset_index(name='tds')

# Show results
scorers.head(50)


2025 done.
Downcasting floats.


,player_id,player,tds
0,00-0030061,Z.Ertz,1
1,00-0030279,K.Allen,1
2,00-0030506,T.Kelce,1
3,00-0030564,D.Hopkins,1
4,00-0032764,D.Henry,2
5,00-0033288,G.Kittle,1
6,00-0033293,A.Jones,1
7,00-0033553,J.Conner,1
8,00-0033858,J.Smith,1
9,00-0033873,P.Mahomes,1


In [154]:
predictions = pd.read_csv('predictions.csv')

#Keep only player_id, player_display_name, predicted_touchdown_probability, and model_edge
predictions = predictions[['player_id', 'player_display_name', 'position','team', 'predicted_touchdown_probability', 'price', 'model_edge']]

#Sort by predicted_touchdown_probability in descending order
predictions = predictions.sort_values(by='predicted_touchdown_probability', ascending=False)

#Show the top 50 players
predictions.head(50)



,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge
0,00-0033553,James Conner,RB,ARI,0.607986,-155.0,0.000143
1,00-0034844,Saquon Barkley,RB,PHI,0.596159,-185.0,-0.052964
2,00-0036223,Jonathan Taylor,RB,IND,0.577282,-180.0,-0.065575
3,00-0035700,Josh Jacobs,RB,GB,0.575883,-160.0,-0.039501
4,00-0037248,James Cook,RB,BUF,0.573437,105.0,0.085632
5,00-0038542,Bijan Robinson,RB,ATL,0.568529,-175.0,-0.067835
6,00-0039139,Jahmyr Gibbs,RB,DET,0.563954,-105.0,0.051759
7,00-0037840,Kyren Williams,RB,LA,0.562856,-140.0,-0.020477
8,00-0032764,Derrick Henry,RB,BAL,0.555453,-145.0,-0.036384
9,00-0036900,Ja'Marr Chase,WR,CIN,0.546660,-130.0,-0.018558


In [155]:
# Join predictions with scorers: prefer player_id, else fall back to name
if 'player_id' in scorers.columns:
    pred_scored = predictions.merge(
        scorers[['player_id', 'tds']], on='player_id', how='inner'
    )
else:
    pred_scored = predictions.merge(
        scorers[['player', 'tds']], left_on='player_display_name', right_on='player', how='inner'
    )

pred_scored.sort_values(['predicted_touchdown_probability'], ascending=[False])

,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge,tds
0,00-0033553,James Conner,RB,ARI,0.607986,-155.0,0.000143,1
1,00-0034844,Saquon Barkley,RB,PHI,0.596159,-185.0,-0.052964,1
2,00-0035700,Josh Jacobs,RB,GB,0.575883,-160.0,-0.039501,1
3,00-0037248,James Cook,RB,BUF,0.573437,105.0,0.085632,1
4,00-0038542,Bijan Robinson,RB,ATL,0.568529,-175.0,-0.067835,1
5,00-0037840,Kyren Williams,RB,LA,0.562856,-140.0,-0.020477,1
6,00-0032764,Derrick Henry,RB,BAL,0.555453,-145.0,-0.036384,2
7,00-0036158,J.K. Dobbins,RB,DEN,0.541170,160.0,0.156555,1
8,00-0039893,Brian Thomas Jr.,WR,JAX,0.522612,130.0,0.087829,1
9,00-0038597,Chase Brown,RB,CIN,0.521273,-150.0,-0.078727,1


In [156]:
###Simulate betting on the top 10 running backs
def simulate_betting(df, scorers):
    stake = 10.0

    bets = df.copy()

    # Merge to mark hits
    bets = bets.merge(
        scorers[['player_id', 'tds']], on='player_id', how='left'
    )
    bets['tds'] = bets['tds'].fillna(0).astype(int)
    bets['hit'] = bets['tds'] > 0

    # American odds payout logic
    # profit_if_win = stake * (odds/100) if odds > 0 else stake * (100/abs(odds))
    # profit_if_loss = -stake
    is_plus = bets['price'] > 0
    profit_if_win = stake * (bets['price'] / 100.0)
    profit_if_win = profit_if_win.where(is_plus, stake * (100.0 / bets['price'].abs()))

    bets['profit'] = np.where(bets['hit'], profit_if_win, -stake)

    # Add total return
    bets['return'] = stake + bets['profit']

    # Summary metrics
    num_bets = len(bets)
    hits = int(bets['hit'].sum())
    hit_rate = hits / num_bets if num_bets else 0.0
    total_profit = float(bets['profit'].sum())
    roi = total_profit / (stake * num_bets) if num_bets else 0.0

    summary = {
        'bets': num_bets,
        'hits': hits,
        'hit_rate': round(hit_rate, 3),
        'total_profit': round(total_profit, 2),
        'roi': round(roi, 3)
    }

    display(summary)

    # Show detailed results
    cols = [
        'player_id', 'player_display_name', 'team', 'position', 'price',
        'predicted_touchdown_probability', 'model_edge', 'tds', 'hit', 'profit', 'return'
    ]
    return bets[cols].sort_values(['predicted_touchdown_probability'], ascending=[False]).reset_index(drop=True)



In [157]:
# output players from predictions that play for 'PHI', 'KC', 'LAC' or 'DAL'

#find players with model_edge > 0 and price < 500
ev = predictions[predictions['model_edge'] > 0.10] 
ev = ev[ev['price'] < 500]

#sort by model_edge in descending order
ev = ev.sort_values('model_edge', ascending=False)
ev







,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge
33,00-0038117,Wan'Dale Robinson,WR,NYG,0.448121,400.0,0.248121
40,00-0038544,Quentin Johnston,WR,LAC,0.426013,370.0,0.213247
48,00-0033858,Jonnu Smith,TE,PIT,0.411840,400.0,0.211840
51,00-0033923,Kareem Hunt,RB,KC,0.406209,370.0,0.193443
39,00-0037256,Rachaad White,RB,TB,0.430675,320.0,0.192580
66,00-0036894,Pat Freiermuth,TE,PIT,0.369121,425.0,0.178645
47,00-0036139,Rico Dowdle,RB,CAR,0.413888,320.0,0.175793
17,00-0037744,Trey McBride,TE,ARI,0.500235,200.0,0.166901
15,00-0036912,DeVonta Smith,WR,PHI,0.515893,180.0,0.158750
10,00-0036158,J.K. Dobbins,RB,DEN,0.541170,160.0,0.156555


In [158]:
simulate_betting(ev, scorers)

{'bets': 24, 'hits': 6, 'hit_rate': 0.25, 'total_profit': -7.5, 'roi': -0.031}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0036158,J.K. Dobbins,DEN,RB,160.0,0.541170,0.156555,1,True,16.0,26.0
1,00-0036912,DeVonta Smith,PHI,WR,180.0,0.515893,0.158750,0,False,-10.0,0.0
2,00-0036875,Rhamondre Stevenson,NE,RB,155.0,0.502357,0.110200,0,False,-10.0,0.0
3,00-0037744,Trey McBride,ARI,TE,200.0,0.500235,0.166901,0,False,-10.0,0.0
4,00-0035676,A.J. Brown,PHI,WR,160.0,0.499252,0.114636,0,False,-10.0,0.0
5,00-0036252,Michael Pittman,IND,WR,215.0,0.450189,0.132728,1,True,21.5,31.5
6,00-0038117,Wan'Dale Robinson,NYG,WR,400.0,0.448121,0.248121,0,False,-10.0,0.0
7,00-0039901,Keon Coleman,BUF,WR,210.0,0.443093,0.120513,1,True,21.0,31.0
8,00-0036407,Jerry Jeudy,CLE,WR,200.0,0.438407,0.105074,0,False,-10.0,0.0
9,00-0037256,Rachaad White,TB,RB,320.0,0.430675,0.192580,0,False,-10.0,0.0


In [159]:
### Get top 10 rb, wr, te, qb from predictions
top_rb = predictions[predictions['position'] == 'RB'].sort_values('predicted_touchdown_probability', ascending=False).head(10)
top_wr = predictions[predictions['position'] == 'WR'].sort_values('predicted_touchdown_probability', ascending=False).head(15)
top_te = predictions[predictions['position'] == 'TE'].sort_values('predicted_touchdown_probability', ascending=False).head(5)
top_qb = predictions[predictions['position'] == 'QB'].sort_values('predicted_touchdown_probability', ascending=False).head(5)

top_rb



,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge
0,00-0033553,James Conner,RB,ARI,0.607986,-155.0,0.000143
1,00-0034844,Saquon Barkley,RB,PHI,0.596159,-185.0,-0.052964
2,00-0036223,Jonathan Taylor,RB,IND,0.577282,-180.0,-0.065575
3,00-0035700,Josh Jacobs,RB,GB,0.575883,-160.0,-0.039501
4,00-0037248,James Cook,RB,BUF,0.573437,105.0,0.085632
5,00-0038542,Bijan Robinson,RB,ATL,0.568529,-175.0,-0.067835
6,00-0039139,Jahmyr Gibbs,RB,DET,0.563954,-105.0,0.051759
7,00-0037840,Kyren Williams,RB,LA,0.562856,-140.0,-0.020477
8,00-0032764,Derrick Henry,RB,BAL,0.555453,-145.0,-0.036384
10,00-0036158,J.K. Dobbins,RB,DEN,0.541170,160.0,0.156555


In [160]:
simulate_betting(top_rb, scorers)

{'bets': 10, 'hits': 8, 'hit_rate': 0.8, 'total_profit': 44.36, 'roi': 0.444}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0033553,James Conner,ARI,RB,-155.0,0.607986,0.000143,1,True,6.451613,16.451613
1,00-0034844,Saquon Barkley,PHI,RB,-185.0,0.596159,-0.052964,1,True,5.405405,15.405405
2,00-0036223,Jonathan Taylor,IND,RB,-180.0,0.577282,-0.065575,0,False,-10.000000,0.000000
3,00-0035700,Josh Jacobs,GB,RB,-160.0,0.575883,-0.039501,1,True,6.250000,16.250000
4,00-0037248,James Cook,BUF,RB,105.0,0.573437,0.085632,1,True,10.500000,20.500000
5,00-0038542,Bijan Robinson,ATL,RB,-175.0,0.568529,-0.067835,1,True,5.714286,15.714286
6,00-0039139,Jahmyr Gibbs,DET,RB,-105.0,0.563954,0.051759,0,False,-10.000000,0.000000
7,00-0037840,Kyren Williams,LA,RB,-140.0,0.562856,-0.020477,1,True,7.142857,17.142857
8,00-0032764,Derrick Henry,BAL,RB,-145.0,0.555453,-0.036384,2,True,6.896552,16.896552
9,00-0036158,J.K. Dobbins,DEN,RB,160.0,0.541170,0.156555,1,True,16.000000,26.000000


In [161]:
#filter top_wr to only include players with model_edge > 0.05 and price < 500
#top_wr = top_wr[top_wr['model_edge'] > 0.05]
#top_wr = top_wr[top_wr['price'] < 500]
simulate_betting(top_wr, scorers)

{'bets': 15,
 'hits': 4,
 'hit_rate': 0.267,
 'total_profit': -41.0,
 'roi': -0.273}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0036900,Ja'Marr Chase,CIN,WR,-130.0,0.546660,-0.018558,0,False,-10.0,0.0
1,00-0039893,Brian Thomas Jr.,JAX,WR,130.0,0.522612,0.087829,1,True,13.0,23.0
2,00-0031408,Mike Evans,TB,WR,110.0,0.522158,0.045967,0,False,-10.0,0.0
3,00-0036912,DeVonta Smith,PHI,WR,180.0,0.515893,0.158750,0,False,-10.0,0.0
4,00-0035676,A.J. Brown,PHI,WR,160.0,0.499252,0.114636,0,False,-10.0,0.0
5,00-0039337,Malik Nabers,NYG,WR,150.0,0.485428,0.085428,0,False,-10.0,0.0
6,00-0031381,Davante Adams,LA,WR,145.0,0.484450,0.076286,0,False,-10.0,0.0
7,00-0039075,Puka Nacua,LA,WR,140.0,0.474064,0.057397,0,False,-10.0,0.0
8,00-0035659,Terry McLaurin,WAS,WR,130.0,0.473641,0.038859,0,False,-10.0,0.0
9,00-0037238,Drake London,ATL,WR,125.0,0.473000,0.028555,0,False,-10.0,0.0


In [162]:
simulate_betting(top_te, scorers)

{'bets': 5, 'hits': 2, 'hit_rate': 0.4, 'total_profit': 26.5, 'roi': 0.53}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0037744,Trey McBride,ARI,TE,200.0,0.500235,0.166901,0,False,-10.0,0.0
1,00-0033885,David Njoku,CLE,TE,230.0,0.423578,0.120547,0,False,-10.0,0.0
2,00-0034753,Mark Andrews,BAL,TE,205.0,0.418387,0.090519,0,False,-10.0,0.0
3,00-0033858,Jonnu Smith,PIT,TE,400.0,0.411840,0.211840,1,True,40.0,50.0
4,00-0030506,Travis Kelce,KC,TE,165.0,0.393427,0.016068,1,True,16.5,26.5


In [163]:
simulate_betting(top_qb, scorers)

{'bets': 5, 'hits': 4, 'hit_rate': 0.8, 'total_profit': 44.5, 'roi': 0.89}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0034857,Josh Allen,BUF,QB,-120.0,0.448580,-0.096874,2,True,8.333333,18.333333
1,00-0039910,Jayden Daniels,WAS,QB,170.0,0.357943,-0.012428,0,False,-10.000000,0.000000
2,00-0036389,Jalen Hurts,PHI,QB,-150.0,0.309135,-0.290865,2,True,6.666667,16.666667
3,00-0035710,Daniel Jones,IND,QB,190.0,0.299345,-0.045482,2,True,19.000000,29.000000
4,00-0034796,Lamar Jackson,BAL,QB,205.0,0.267586,-0.060283,1,True,20.500000,30.500000


In [164]:
top_predictors = predictions[predictions['predicted_touchdown_probability'] >= 0.50]
simulate_betting(top_predictors, scorers)

{'bets': 18,
 'hits': 11,
 'hit_rate': 0.611,
 'total_profit': 21.17,
 'roi': 0.118}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0033553,James Conner,ARI,RB,-155.0,0.607986,0.000143,1,True,6.451613,16.451613
1,00-0034844,Saquon Barkley,PHI,RB,-185.0,0.596159,-0.052964,1,True,5.405405,15.405405
2,00-0036223,Jonathan Taylor,IND,RB,-180.0,0.577282,-0.065575,0,False,-10.000000,0.000000
3,00-0035700,Josh Jacobs,GB,RB,-160.0,0.575883,-0.039501,1,True,6.250000,16.250000
4,00-0037248,James Cook,BUF,RB,105.0,0.573437,0.085632,1,True,10.500000,20.500000
5,00-0038542,Bijan Robinson,ATL,RB,-175.0,0.568529,-0.067835,1,True,5.714286,15.714286
6,00-0039139,Jahmyr Gibbs,DET,RB,-105.0,0.563954,0.051759,0,False,-10.000000,0.000000
7,00-0037840,Kyren Williams,LA,RB,-140.0,0.562856,-0.020477,1,True,7.142857,17.142857
8,00-0032764,Derrick Henry,BAL,RB,-145.0,0.555453,-0.036384,2,True,6.896552,16.896552
9,00-0036900,Ja'Marr Chase,CIN,WR,-130.0,0.546660,-0.018558,0,False,-10.000000,0.000000
